# Evaluation code for the Singapore Tourism Agent

This notebook extracts the evaluation stages from the supplied project script. Run the project setup, corpus/FAISS construction, four tool definitions, and Agent creation in this same runtime first. Required objects include `PROJECT_DIR`, `tourism_agent`, `OFFICIAL_DOMAINS`, `TOOL_AUDIT_LOG`, `openrouter_client`, and `OPENROUTER_MODEL`.

Run stages 1–3. Then **stop and complete the human review spreadsheet** before stages 4–5. Do not use “Run all” on the first run. Agent and judge calls use the API and can incur charges.

Two documented changes: tool sets accept commas, pipes, and semicolons; an existing human review file is preserved. Scoring prompts, validator rules, model selection, and archive logic are retained from the original script. The original extraction is also supplied for historical reproducibility.

The judge receives the question, expected answer/rubric, expected source title/URL, abstention flag, and candidate answer. It does not receive the full retrieved page text; its faithfulness and citation scores are judgments, not independently verified source checks.

## 1. Answer validation and asynchronous Agent execution

Requires the existing Agent and official-domain list.

In [ ]:
# ============================================================
# Agent answer validation + asynchronous execution
# Agent回答验证＋异步运行
# ============================================================

import re
import time
import json

from urllib.parse import urlparse

import pandas as pd

from agents import Runner


# ============================================================
# 1. 初始化工具审计日志
# ============================================================

# 如果之前已经创建过日志，就保留原对象；
# 如果尚未创建，则在这里创建。
if "TOOL_AUDIT_LOG" not in globals():
    TOOL_AUDIT_LOG = []


# ============================================================
# 2. 检查前面的关键变量
# ============================================================

required_variables = [
    "tourism_agent",
    "OFFICIAL_DOMAINS",
]

missing_variables = [
    variable_name
    for variable_name in required_variables
    if variable_name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "以下变量尚未创建："
        + ", ".join(missing_variables)
        + "。\n请先依次运行：\n"
        "1. OpenRouter配置单元格；\n"
        "2. 知识库和FAISS索引单元格；\n"
        "3. Agent工具定义单元格；\n"
        "4. tourism_agent创建单元格。"
    )


# ============================================================
# 3. URL和日期识别规则
# ============================================================

URL_PATTERN = re.compile(
    r"https?://[^\s\]\[()<>{}\"']+"
)

DATE_PATTERN = re.compile(
    r"\b20\d{2}-\d{2}-\d{2}\b"
)


# ============================================================
# 4. 清理从回答中提取的网址
# ============================================================

def clean_extracted_url(url):
    """
    清除网址末尾可能出现的英文或中文标点符号。
    """

    return str(url).strip().rstrip(
        ".,;:!?)]}>\"'，。；：！？）】》"
    )


# ============================================================
# 5. 统一域名格式
# ============================================================

def normalize_domain(domain):
    """
    将域名统一为小写并移除www.。

    兼容以下输入：
    - www.mandai.com
    - mandai.com
    - https://www.mandai.com/example
    """

    domain = str(domain or "").strip().lower()

    if not domain:
        return ""

    if "://" in domain:
        domain = (
            urlparse(domain).hostname
            or ""
        ).lower()

    # 删除端口号
    domain = domain.split(":")[0]

    if domain.startswith("www."):
        domain = domain[4:]

    return domain


# 将官方域名列表统一格式
NORMALIZED_OFFICIAL_DOMAINS = sorted({
    normalize_domain(domain)
    for domain in OFFICIAL_DOMAINS
    if normalize_domain(domain)
})


# ============================================================
# 6. 判断网址是否属于官方允许域名
# ============================================================

def host_is_allowed(url):
    """
    判断网址是否属于OFFICIAL_DOMAINS中的官方域名。

    例如：
    - mandai.com
    - www.mandai.com
    - tickets.mandai.com

    都可以被识别为mandai.com体系内的网址。
    """

    try:
        cleaned_url = clean_extracted_url(url)

        parsed_url = urlparse(
            cleaned_url
        )

        host = normalize_domain(
            parsed_url.hostname
            or ""
        )

        if not host:
            return False

        return any(
            host == allowed_domain
            or host.endswith(
                "." + allowed_domain
            )
            for allowed_domain
            in NORMALIZED_OFFICIAL_DOMAINS
        )

    except Exception:
        return False


# ============================================================
# 7. 验证Agent最终回答
# ============================================================

def validate_agent_answer(
    answer,
    audit_log,
):
    """
    验证Agent回答是否：

    1. 调用了本地知识库；
    2. 提供了来源网址；
    3. 所有来源都属于官方允许域名；
    4. 提供了证据日期；
    5. 记录了实际调用过的工具。
    """

    answer = str(
        answer
        or ""
    ).strip()

    audit_log = (
        audit_log
        or []
    )

    # 从回答中提取网址
    raw_urls = URL_PATTERN.findall(
        answer
    )

    urls = sorted({
        clean_extracted_url(url)
        for url in raw_urls
        if clean_extracted_url(url)
    })

    # 从审计日志中读取实际调用过的工具
    tools_called = []

    for entry in audit_log:
        if not isinstance(entry, dict):
            continue

        tool_name = (
            entry.get("tool")
            or entry.get("tool_name")
            or ""
        )

        if tool_name:
            tools_called.append(
                str(tool_name)
            )

    checks = {
        # 是否调用本地RAG知识库
        "used_local_search": (
            "search_knowledge_base"
            in tools_called
        ),

        # 是否调用路线工具
        "used_route_tool": (
            "get_route_information"
            in tools_called
        ),

        # 是否调用实时官方网页工具
        "used_live_official_page": (
            "fetch_live_official_page"
            in tools_called
        ),

        # 回答中是否出现来源网址
        "has_source_url": bool(urls),

        # 回答中的全部网址是否属于官方域名
        "all_urls_official": (
            bool(urls)
            and all(
                host_is_allowed(url)
                for url in urls
            )
        ),

        # 回答中是否包含YYYY-MM-DD格式的证据日期
        "has_evidence_date": bool(
            DATE_PATTERN.search(answer)
        ),

        # 记录调用过的工具和提取的网址
        "tools_called": tools_called,
        "urls": urls,
    }

    # 基本通过条件
    #
    # 路线工具和实时网页工具不是每个问题都必须调用，
    # 所以不放进基本通过条件。
    checks["passed"] = all([
        checks["used_local_search"],
        checks["has_source_url"],
        checks["all_urls_official"],
        checks["has_evidence_date"],
    ])

    return checks


# ============================================================
# 8. 异步运行Agent
# ============================================================

async def ask_agent(question):
    """
    在Google Colab中异步运行旅游Agent。

    注意：
    调用这个函数时必须使用：

    result = await ask_agent("问题")

    不能写成：

    result = ask_agent("问题")
    """

    if not question:
        raise ValueError(
            "Question cannot be empty."
        )

    question = str(
        question
    ).strip()

    if not question:
        raise ValueError(
            "Question cannot be empty."
        )

    # 每次开始新问题前清空审计日志
    TOOL_AUDIT_LOG.clear()

    started = time.time()

    try:
        # Colab已经运行事件循环，
        # 因此这里必须使用await Runner.run
        result = await Runner.run(
            tourism_agent,
            input=question,
        )

    except Exception as exc:
        error_name = type(
            exc
        ).__name__

        error_message = str(
            exc
        )

        raise RuntimeError(
            "Agent请求失败。\n\n"
            f"错误类型：{error_name}\n"
            f"错误信息：{error_message}\n\n"
            "请检查：\n"
            "1. Colab Secrets中的名称是否为"
            "OPENROUTER_API_KEY；\n"
            "2. OpenRouter密钥是否有效；\n"
            "3. OpenRouter账户是否有可用额度；\n"
            "4. OPENROUTER_MODEL是否支持工具调用；\n"
            "5. 创建Agent时是否使用model=agent_model；\n"
            "6. 是否已经运行FAISS和工具定义单元格；\n"
            "7. 是否已经重新运行tourism_agent创建单元格。"
        ) from exc

    answer = str(
        result.final_output
        or ""
    ).strip()

    validation = validate_agent_answer(
        answer,
        TOOL_AUDIT_LOG,
    )

    completed_result = {
        "question": question,
        "answer": answer,
        "validation": validation,
        "audit": list(
            TOOL_AUDIT_LOG
        ),
        "elapsed_seconds": round(
            time.time() - started,
            2,
        ),
    }

    return completed_result


# ============================================================
# 9. 单元格加载结果
# ============================================================

print(
    "Agent validation and asynchronous "
    "execution functions loaded successfully."
)

print(
    "Official domains:",
    NORMALIZED_OFFICIAL_DOMAINS,
)

print(
    "使用方法："
    "test = await ask_agent('Your question')"
)

## 2. Load the XLSX and evaluate 30 questions

Input: `PROJECT_DIR / evaluation_questions_30.xlsx`. Writes `evaluation_results_agent.csv` and `.xlsx`. Uses `OPENROUTER_MODEL` as the judge model.

In [ ]:
# ============================================================
# 30-question Agent evaluation
# 从Google Drive读取XLSX并进行异步评估
# ============================================================

import re
import json
import asyncio
from pathlib import Path

import pandas as pd


# ------------------------------------------------------------
# 1. Excel文件路径
# ------------------------------------------------------------

eval_path = PROJECT_DIR / "evaluation_questions_30.xlsx"

if not eval_path.exists():
    raise FileNotFoundError(
        f"找不到评估文件：{eval_path}\n"
        "请确认文件已经保存为真正的XLSX文件，"
        "而不是Google Sheets的.gsheet快捷方式。"
    )

print("Evaluation file:", eval_path)


# ------------------------------------------------------------
# 2. 读取Excel
# ------------------------------------------------------------

questions = pd.read_excel(
    eval_path,
    engine="openpyxl",
)

# 清理列名两侧空格
questions.columns = [
    str(column).strip()
    for column in questions.columns
]

required_columns = [
    "question_id",
    "category",
    "question",
    "expected_answer_from_snapshot",
    "expected_url_or_title",
    "expected_tools",
    "should_abstain",
    "time_sensitive",
    "notes",
]

missing_columns = [
    column
    for column in required_columns
    if column not in questions.columns
]

if missing_columns:
    raise RuntimeError(
        "Excel缺少以下列："
        + ", ".join(missing_columns)
    )

if len(questions) != 30:
    raise RuntimeError(
        f"评估集必须正好有30题，目前读取到{len(questions)}题。"
    )

if not questions["question"].fillna("").astype(str).str.strip().ne("").all():
    empty_rows = (
        questions.index[
            questions["question"]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq("")
        ]
        + 2
    ).tolist()

    raise RuntimeError(
        "以下Excel行的question为空："
        + ", ".join(map(str, empty_rows))
    )

print(f"Successfully loaded {len(questions)} questions.")
display(questions.head())


# ------------------------------------------------------------
# 3. 辅助函数
# ------------------------------------------------------------

def clean_cell(value):
    """把Excel空值转换为空字符串。"""

    if pd.isna(value):
        return ""

    return str(value).strip()


def parse_tool_set(value):
    """
    同时支持以下分隔符：
    search_knowledge_base | search_live_official_web
    search_knowledge_base, search_live_official_web
    search_knowledge_base; search_live_official_web
    """

    value = clean_cell(value)

    return {
        tool_name.strip()
        for tool_name in re.split(r"[,|;]", value)
        if tool_name.strip()
    }


def extract_json_object(text):
    """从模型输出中提取JSON对象。"""

    text = str(text or "").strip()

    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\s*```$",
        "",
        text,
    )

    try:
        return json.loads(text)

    except json.JSONDecodeError:
        match = re.search(
            r"\{.*\}",
            text,
            flags=re.DOTALL,
        )

        if match:
            return json.loads(match.group(0))

        raise


# ------------------------------------------------------------
# 4. OpenRouter LLM Judge
# ------------------------------------------------------------

async def judge_agent_answer(
    question,
    expected,
    expected_url_or_title,
    should_abstain,
    answer,
):
    """
    使用OpenRouter评估Agent回答。

    expected是评估标准或参考答案，不要求Agent逐字匹配。
    """

    question = clean_cell(question)
    expected = clean_cell(expected)
    expected_url_or_title = clean_cell(
        expected_url_or_title
    )
    answer = clean_cell(answer)

    prompt = f"""
Return one valid JSON object only.

Use exactly these keys:
{{
  "faithful_to_available_evidence": 0,
  "correct_against_snapshot": 0,
  "citation_support": 0,
  "appropriate_uncertainty": 0,
  "constraint_satisfaction": 0,
  "concise_reason": ""
}}

Scoring instructions:

1. Each numerical field must be either 0 or 1.
2. The expected answer may be a reference answer or an evaluation rubric.
3. It is not necessarily the only acceptable answer.
4. Multiple different itineraries or recommendations may be correct.
5. Do not penalize a candidate only because it uses a different reasonable
   itinerary, attraction, route or ordering.
6. Evaluate whether factual claims are supported by the available evidence.
7. Penalize invented opening hours, prices, routes, accessibility features,
   closures or unsupported claims.
8. Citation support is 1 only when the cited official sources reasonably
   support the main factual claims.
9. Appropriate uncertainty is 1 when the answer clearly acknowledges missing,
   conflicting, outdated or insufficient evidence when necessary.
10. Constraint satisfaction is 1 when the candidate satisfies the important
    requirements in the question and evaluation criteria.
11. If should_abstain is true, a careful evidence-based refusal may be correct.

Question:
{question}

Expected answer or evaluation criteria:
{expected}

Expected official URL or source title:
{expected_url_or_title}

Should the Agent abstain:
{should_abstain}

Candidate answer:
{answer}
"""

    try:
        response = await openrouter_client.chat.completions.create(
            model=OPENROUTER_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are an evaluation judge. "
                        "Return valid JSON only."
                    ),
                },
                {
                    "role": "user",
                    "content": prompt,
                },
            ],
            temperature=0,
        )

        raw = (
            response.choices[0]
            .message.content
            or ""
        ).strip()

        result = extract_json_object(raw)

        # 保证评分字段只能是0或1
        score_fields = [
            "faithful_to_available_evidence",
            "correct_against_snapshot",
            "citation_support",
            "appropriate_uncertainty",
            "constraint_satisfaction",
        ]

        for field in score_fields:
            value = result.get(field)

            try:
                result[field] = 1 if int(value) == 1 else 0
            except Exception:
                result[field] = None

        return result

    except Exception as exc:
        return {
            "faithful_to_available_evidence": None,
            "correct_against_snapshot": None,
            "citation_support": None,
            "appropriate_uncertainty": None,
            "constraint_satisfaction": None,
            "concise_reason": (
                "Judge failed: "
                f"{type(exc).__name__}: {exc}"
            ),
        }


# ------------------------------------------------------------
# 5. 开始评估
# ------------------------------------------------------------

evaluation_rows = []

# 测试时可以改成2；正式评估改成30
EVALUATION_LIMIT = 30

for row_number, (_, item) in enumerate(
    questions.head(EVALUATION_LIMIT).iterrows(),
    start=1,
):
    question_id = clean_cell(
        item["question_id"]
    )

    question = clean_cell(
        item["question"]
    )

    expected = clean_cell(
        item["expected_answer_from_snapshot"]
    )

    expected_url_or_title = clean_cell(
        item["expected_url_or_title"]
    )

    expected_tools = parse_tool_set(
        item["expected_tools"]
    )

    should_abstain = clean_cell(
        item["should_abstain"]
    )

    print(
        f"[{row_number}/{EVALUATION_LIMIT}] "
        f"{question_id}: {question}"
    )

    try:
        # ask_agent是异步函数，因此必须使用await
        generated = await ask_agent(
            question
        )

        answer = clean_cell(
            generated.get("answer", "")
        )

        validation = (
            generated.get("validation", {})
            or {}
        )

        tools_called_list = (
            validation.get("tools_called", [])
            or []
        )

        actual_tools = {
            str(tool).strip()
            for tool in tools_called_list
            if str(tool).strip()
        }

        # 只要求预期工具是实际工具的子集
        # Agent可以合理地使用额外工具
        tool_selection_correct = (
            expected_tools.issubset(
                actual_tools
            )
        )

        judge = await judge_agent_answer(
            question=question,
            expected=expected,
            expected_url_or_title=(
                expected_url_or_title
            ),
            should_abstain=should_abstain,
            answer=answer,
        )

        evaluation_rows.append({
            **item.to_dict(),

            "answer": answer,

            "actual_tools": " | ".join(
                sorted(actual_tools)
            ),

            "tool_selection_correct": (
                tool_selection_correct
            ),

            "validator_passed": validation.get(
                "passed"
            ),

            "judge_faithful": judge.get(
                "faithful_to_available_evidence"
            ),

            "judge_snapshot_correct": judge.get(
                "correct_against_snapshot"
            ),

            "judge_citation_support": judge.get(
                "citation_support"
            ),

            "judge_uncertainty": judge.get(
                "appropriate_uncertainty"
            ),

            "judge_constraint_satisfaction": judge.get(
                "constraint_satisfaction"
            ),

            "judge_reason": judge.get(
                "concise_reason"
            ),

            "evaluation_error": "",
        })

    except Exception as exc:
        # 一题失败时继续评估下一题
        evaluation_rows.append({
            **item.to_dict(),

            "answer": "",

            "actual_tools": "",

            "tool_selection_correct": False,

            "validator_passed": False,

            "judge_faithful": None,

            "judge_snapshot_correct": None,

            "judge_citation_support": None,

            "judge_uncertainty": None,

            "judge_constraint_satisfaction": None,

            "judge_reason": "",

            "evaluation_error": (
                f"{type(exc).__name__}: {exc}"
            ),
        })

        print(
            f"Question {question_id} failed: "
            f"{type(exc).__name__}: {exc}"
        )

    # 减少OpenRouter连续请求过快的问题
    await asyncio.sleep(1)


# ------------------------------------------------------------
# 6. 保存结果
# ------------------------------------------------------------

evaluation = pd.DataFrame(
    evaluation_rows
)

csv_output_path = (
    PROJECT_DIR
    / "evaluation_results_agent.csv"
)

xlsx_output_path = (
    PROJECT_DIR
    / "evaluation_results_agent.xlsx"
)

evaluation.to_csv(
    csv_output_path,
    index=False,
    encoding="utf-8-sig",
)

evaluation.to_excel(
    xlsx_output_path,
    index=False,
    engine="openpyxl",
)

print("\nEvaluation completed.")
print("CSV result:", csv_output_path)
print("XLSX result:", xlsx_output_path)


# ------------------------------------------------------------
# 7. 显示结果
# ------------------------------------------------------------

display(
    evaluation.head()
)

score_columns = [
    "tool_selection_correct",
    "validator_passed",
    "judge_faithful",
    "judge_snapshot_correct",
    "judge_citation_support",
    "judge_uncertainty",
    "judge_constraint_satisfaction",
]

display(
    evaluation[
        score_columns
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .mean()
    .to_frame("mean")
)

## 3. Prepare ten human-review records

After this stage, open `human_review_10_with_disagreements.xlsx`, enter 0/1 scores or N/A, and save it. If a file already exists, verify its answers belong to this run.

In [ ]:
REVIEW_IDS = [
    "Q01", "Q06",
    "Q09", "Q13",
    "Q15", "Q19",
    "Q21", "Q24",
    "Q25",
    "Q29",
]

human_review = evaluation[
    evaluation["question_id"]
    .astype(str)
    .str.strip()
    .isin(REVIEW_IDS)
].copy()

# 按指定顺序排列
human_review["review_order"] = pd.Categorical(
    human_review["question_id"],
    categories=REVIEW_IDS,
    ordered=True,
)

human_review = (
    human_review
    .sort_values("review_order")
    .drop(columns=["review_order"])
)

# 添加人工评分列
human_review["human_tool_selection_correct"] = ""
human_review["human_faithful"] = ""
human_review["human_snapshot_correct"] = ""
human_review["human_citation_support"] = ""
human_review["human_uncertainty"] = ""
human_review["human_constraint_satisfaction"] = ""
human_review["human_overall_pass"] = ""
human_review["human_notes"] = ""

human_review_path = (
    PROJECT_DIR / "human_review_10_with_disagreements.xlsx"
)

if human_review_path.exists():
    print("Existing human review preserved:", human_review_path)
    print("Do not use it for a new run unless its answers match this evaluation.")
else:
    human_review.to_excel(
        human_review_path,
        index=False,
        engine="openpyxl",
    )

print("人工复核表已生成：", human_review_path)

display(
    human_review[
        [
            "question_id",
            "category",
            "question",
            "answer",
            "actual_tools",
            "judge_reason",
        ]
    ]
)

## 4. Summarize completed human review and judge disagreements

Run only after the human review file has been completed. This stage writes the summary and disagreement columns.

In [ ]:
# ============================================================
# 最终汇总：自动评估 + 人工复核 + Judge与人工一致性
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# 1. 读取人工复核文件
# ------------------------------------------------------------

review_path = (
    PROJECT_DIR
    / "human_review_10_with_disagreements.xlsx"
)

# 如果文件使用其他名称，在这里修改
if not review_path.exists():
    alternative_path = (
        PROJECT_DIR
        / "human_review_10_with_disagreements.xlsx"
    )

    if alternative_path.exists():
        review_path = alternative_path
    else:
        raise FileNotFoundError(
            "没有找到人工复核文件："
            "human_review_10_completed.xlsx "
            "或 human_review_10_with_disagreements.xlsx"
        )

review = pd.read_excel(
    review_path,
    engine="openpyxl",
)

print("Human review file:", review_path)


# ------------------------------------------------------------
# 2. 把评分列转换成数字
# N/A会自动转换为缺失值NaN
# ------------------------------------------------------------

human_score_columns = [
    "human_tool_selection_correct",
    "human_faithful",
    "human_snapshot_correct",
    "human_citation_support",
    "human_uncertainty",
    "human_constraint_satisfaction",
    "human_overall_pass",
]

judge_score_columns = [
    "judge_faithful",
    "judge_snapshot_correct",
    "judge_citation_support",
    "judge_uncertainty",
    "judge_constraint_satisfaction",
]

for column in (
    human_score_columns
    + judge_score_columns
):
    if column in review.columns:
        review[column] = pd.to_numeric(
            review[column],
            errors="coerce",
        )


# ------------------------------------------------------------
# 3. 比较Judge和人工评分
# 只有双方都有有效评分时才进行比较
# ------------------------------------------------------------

comparison_pairs = {
    "faithfulness_disagreement": (
        "judge_faithful",
        "human_faithful",
    ),

    "snapshot_disagreement": (
        "judge_snapshot_correct",
        "human_snapshot_correct",
    ),

    "citation_disagreement": (
        "judge_citation_support",
        "human_citation_support",
    ),

    "uncertainty_disagreement": (
        "judge_uncertainty",
        "human_uncertainty",
    ),

    "constraint_disagreement": (
        "judge_constraint_satisfaction",
        "human_constraint_satisfaction",
    ),
}

for disagreement_column, (
    judge_column,
    human_column,
) in comparison_pairs.items():

    valid = (
        review[judge_column].notna()
        & review[human_column].notna()
    )

    # 没有有效评分的题设为NA，而不是错误地判为分歧
    review[disagreement_column] = pd.NA

    review.loc[
        valid,
        disagreement_column,
    ] = (
        review.loc[
            valid,
            judge_column,
        ]
        != review.loc[
            valid,
            human_column,
        ]
    )


# ------------------------------------------------------------
# 4. 计算自动评估汇总
# ------------------------------------------------------------

automatic_metrics = [
    "tool_selection_correct",
    "validator_passed",
    "judge_faithful",
    "judge_snapshot_correct",
    "judge_citation_support",
    "judge_uncertainty",
    "judge_constraint_satisfaction",
]

summary_rows = []

for metric in automatic_metrics:
    if metric not in evaluation.columns:
        continue

    values = pd.to_numeric(
        evaluation[metric],
        errors="coerce",
    )

    summary_rows.append({
        "evaluation_source": "automatic",
        "metric": metric,
        "sample_size": int(
            values.notna().sum()
        ),
        "passed": int(
            values.sum()
        ),
        "rate": values.mean(),
    })


# ------------------------------------------------------------
# 5. 计算人工复核汇总
# ------------------------------------------------------------

for metric in human_score_columns:
    if metric not in review.columns:
        continue

    values = pd.to_numeric(
        review[metric],
        errors="coerce",
    )

    summary_rows.append({
        "evaluation_source": "human",
        "metric": metric,
        "sample_size": int(
            values.notna().sum()
        ),
        "passed": int(
            values.sum()
        ),
        "rate": values.mean(),
    })


# ------------------------------------------------------------
# 6. 计算Judge与人工分歧率
# ------------------------------------------------------------

for metric in comparison_pairs:
    valid_values = review[metric].dropna()

    summary_rows.append({
        "evaluation_source": (
            "judge_human_comparison"
        ),
        "metric": metric,
        "sample_size": len(
            valid_values
        ),
        "passed": int(
            valid_values.astype(bool).sum()
        ),
        "rate": (
            valid_values.astype(bool).mean()
            if len(valid_values)
            else None
        ),
    })


summary = pd.DataFrame(
    summary_rows
)


# ------------------------------------------------------------
# 7. 保存结果
# ------------------------------------------------------------

summary_path = (
    PROJECT_DIR
    / "evaluation_summary_agent.csv"
)

review_output_path = (
    PROJECT_DIR
    / "human_review_10_with_disagreements.xlsx"
)

summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

review.to_excel(
    review_output_path,
    index=False,
    engine="openpyxl",
)


# ------------------------------------------------------------
# 8. 显示汇总
# ------------------------------------------------------------

display(summary)


# ------------------------------------------------------------
# 9. 显示Judge与人工存在分歧的题目
# ------------------------------------------------------------

disagreement_columns = list(
    comparison_pairs.keys()
)

has_disagreement = (
    review[
        disagreement_columns
    ]
    .fillna(False)
    .astype(bool)
    .any(axis=1)
)

disagreements = review.loc[
    has_disagreement,
    [
        "question_id",
        "question",
        "judge_reason",
        "human_notes",
        *disagreement_columns,
    ],
]

print(
    "Questions with judge-human disagreements:",
    len(disagreements),
)

display(disagreements)

print("Summary saved to:", summary_path)
print("Comparison saved to:", review_output_path)

## 5. Archive the experiment (original GPT-6.1 saving code)

Check the actual configured model before running this stage. The saving code retains its GPT-6.1 naming and metadata logic. It saves available results; it does not rerun the evaluation.

In [ ]:
# ============================================================
# Save complete GPT-6.1 experiment
# 保存GPT-6.1自动评估、人工复核、配置和日志
# ============================================================

import re
import json
import shutil
import platform
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd


# ============================================================
# 1. 检查必要变量
# ============================================================

required_variables = [
    "PROJECT_DIR",
    "evaluation",
]

missing_variables = [
    variable_name
    for variable_name in required_variables
    if variable_name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "请先运行GPT-6.1评估代码。缺少变量："
        + ", ".join(missing_variables)
    )


PROJECT_DIR = Path(PROJECT_DIR)

if not isinstance(evaluation, pd.DataFrame):
    raise TypeError(
        "evaluation必须是一个pandas DataFrame。"
    )

if evaluation.empty:
    raise RuntimeError(
        "evaluation为空，请先完成GPT-6.1的30题评估。"
    )


# ============================================================
# 2. 模型和实验名称
# ============================================================

TESTED_MODEL = globals().get(
    "GENERATION_MODEL",
    globals().get(
        "OPENROUTER_MODEL",
        "openai/gpt-6.1-sol-pro",
    ),
)

MODEL_FILE_LABEL = "gpt6_1"

RUN_TIMESTAMP_UTC = (
    datetime.now(timezone.utc)
    .strftime("%Y%m%d_%H%M%S")
)

EXPERIMENT_NAME = (
    f"{MODEL_FILE_LABEL}_baseline_"
    f"{RUN_TIMESTAMP_UTC}"
)

EXPERIMENTS_DIR = (
    PROJECT_DIR / "experiments"
)

RUN_DIR = (
    EXPERIMENTS_DIR / EXPERIMENT_NAME
)

EXPERIMENTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

print("Tested model:", TESTED_MODEL)
print("Experiment folder:", RUN_DIR)


# ============================================================
# 3. 通用评分转换函数
# 支持0/1、TRUE/FALSE、Yes/No和Pass/Fail
# ============================================================

def parse_binary_score(value):

    if value is None or pd.isna(value):
        return pd.NA

    if isinstance(value, bool):
        return int(value)

    if isinstance(value, (int, float)):

        numeric_value = float(value)

        if numeric_value == 1.0:
            return 1

        if numeric_value == 0.0:
            return 0

        return pd.NA

    text = str(value).strip().lower()

    score_mapping = {
        "1": 1,
        "1.0": 1,
        "true": 1,
        "yes": 1,
        "y": 1,
        "pass": 1,
        "passed": 1,
        "correct": 1,

        "0": 0,
        "0.0": 0,
        "false": 0,
        "no": 0,
        "n": 0,
        "fail": 0,
        "failed": 0,
        "incorrect": 0,

        "": pd.NA,
        "na": pd.NA,
        "n/a": pd.NA,
        "none": pd.NA,
        "null": pd.NA,
        "not applicable": pd.NA,
    }

    return score_mapping.get(
        text,
        pd.NA,
    )


# ============================================================
# 4. 工具集合解析函数
# 支持逗号、竖线和分号
# ============================================================

def parse_tool_set(value):

    if value is None or pd.isna(value):
        return set()

    if isinstance(
        value,
        (list, tuple, set),
    ):
        return {
            str(tool_name).strip()
            for tool_name in value
            if str(tool_name).strip()
        }

    return {
        tool_name.strip()
        for tool_name in re.split(
            r"\s*[,|;]\s*",
            str(value),
        )
        if tool_name.strip()
    }


# ============================================================
# 5. 保存原始自动评估结果
# ============================================================

evaluation_raw = evaluation.copy()

raw_csv_path = (
    RUN_DIR
    / f"evaluation_results_{MODEL_FILE_LABEL}_raw.csv"
)

raw_xlsx_path = (
    RUN_DIR
    / f"evaluation_results_{MODEL_FILE_LABEL}_raw.xlsx"
)

evaluation_raw.to_csv(
    raw_csv_path,
    index=False,
    encoding="utf-8-sig",
)

evaluation_raw.to_excel(
    raw_xlsx_path,
    index=False,
)

print("Saved raw evaluation results.")


# ============================================================
# 6. 修正工具选择判断
#
# 判定规则：
# - 预期工具必须全部出现在实际工具中
# - 工具顺序不重要
# - 允许Agent调用额外工具
# ============================================================

evaluation_corrected = (
    evaluation_raw.copy()
)

required_tool_columns = {
    "expected_tools",
    "actual_tools",
}

if required_tool_columns.issubset(
    evaluation_corrected.columns
):

    evaluation_corrected[
        "tool_selection_correct"
    ] = evaluation_corrected.apply(
        lambda row: parse_tool_set(
            row["expected_tools"]
        ).issubset(
            parse_tool_set(
                row["actual_tools"]
            )
        ),
        axis=1,
    )

else:
    print(
        "警告：没有找到expected_tools或actual_tools，"
        "无法重新计算tool_selection_correct。"
    )


corrected_csv_path = (
    RUN_DIR
    / f"evaluation_results_{MODEL_FILE_LABEL}_corrected.csv"
)

corrected_xlsx_path = (
    RUN_DIR
    / f"evaluation_results_{MODEL_FILE_LABEL}_corrected.xlsx"
)

evaluation_corrected.to_csv(
    corrected_csv_path,
    index=False,
    encoding="utf-8-sig",
)

evaluation_corrected.to_excel(
    corrected_xlsx_path,
    index=False,
)

print("Saved corrected evaluation results.")


# ============================================================
# 7. 读取或提取30题评估集
# ============================================================

if (
    "questions" in globals()
    and isinstance(
        questions,
        pd.DataFrame,
    )
):

    evaluation_questions = (
        questions.copy()
    )

elif "eval_path" in globals():

    evaluation_questions_path = Path(
        eval_path
    )

    if not evaluation_questions_path.exists():
        raise FileNotFoundError(
            "找不到评估题文件："
            f"{evaluation_questions_path}"
        )

    file_suffix = (
        evaluation_questions_path
        .suffix
        .lower()
    )

    if file_suffix in {
        ".xlsx",
        ".xls",
    }:
        evaluation_questions = (
            pd.read_excel(
                evaluation_questions_path
            )
        )
    else:
        evaluation_questions = (
            pd.read_csv(
                evaluation_questions_path
            )
        )

else:

    question_columns = [
        "question_id",
        "category",
        "question",
        "expected_answer_from_snapshot",
        "expected_url_or_title",
        "expected_tools",
        "should_abstain",
        "time_sensitive",
        "notes",
    ]

    available_question_columns = [
        column_name
        for column_name in question_columns
        if column_name
        in evaluation_corrected.columns
    ]

    evaluation_questions = (
        evaluation_corrected[
            available_question_columns
        ]
        .copy()
    )


questions_csv_path = (
    RUN_DIR
    / "evaluation_questions_30.csv"
)

questions_xlsx_path = (
    RUN_DIR
    / "evaluation_questions_30.xlsx"
)

evaluation_questions.to_csv(
    questions_csv_path,
    index=False,
    encoding="utf-8-sig",
)

evaluation_questions.to_excel(
    questions_xlsx_path,
    index=False,
)

print(
    "Saved evaluation questions:",
    len(evaluation_questions),
)


# ============================================================
# 8. 生成自动评估摘要
# ============================================================

automatic_metric_columns = [
    "tool_selection_correct",
    "validator_passed",
    "judge_faithful",
    "judge_snapshot_correct",
    "judge_citation_support",
    "judge_uncertainty",
    "judge_constraint_satisfaction",
]

automatic_summary_rows = []

for metric in automatic_metric_columns:

    if metric not in evaluation_corrected.columns:
        print(
            f"Automatic metric not found: {metric}"
        )
        continue

    converted_values = (
        evaluation_corrected[metric]
        .apply(parse_binary_score)
        .dropna()
        .astype(int)
    )

    if converted_values.empty:
        print(
            f"No valid automatic scores: {metric}"
        )
        continue

    automatic_summary_rows.append({
        "metric": metric,
        "valid_questions": int(
            len(converted_values)
        ),
        "passed_questions": int(
            converted_values.sum()
        ),
        "pass_rate": float(
            converted_values.mean()
        ),
        "tested_model": TESTED_MODEL,
    })


automatic_summary = pd.DataFrame(
    automatic_summary_rows
)

automatic_summary_csv_path = (
    RUN_DIR
    / f"automatic_summary_{MODEL_FILE_LABEL}.csv"
)

automatic_summary_xlsx_path = (
    RUN_DIR
    / f"automatic_summary_{MODEL_FILE_LABEL}.xlsx"
)

automatic_summary.to_csv(
    automatic_summary_csv_path,
    index=False,
    encoding="utf-8-sig",
)

automatic_summary.to_excel(
    automatic_summary_xlsx_path,
    index=False,
)

print("\nAutomatic evaluation summary:")
display(automatic_summary)


# ============================================================
# 9. 读取填写完成的人工复核文件
# ============================================================

HUMAN_REVIEW_PATH = (
    PROJECT_DIR
    / "human_review_10_with_disagreements.xlsx"
)

if not HUMAN_REVIEW_PATH.exists():
    raise FileNotFoundError(
        "找不到人工复核文件：\n"
        f"{HUMAN_REVIEW_PATH}\n\n"
        "请确认该文件已经保存到PROJECT_DIR。"
    )


human_review_dataframe = pd.read_excel(
    HUMAN_REVIEW_PATH
)

human_review_dataframe.columns = (
    human_review_dataframe.columns
    .astype(str)
    .str.strip()
)

print("\nHuman-review source file:")
print(HUMAN_REVIEW_PATH)

print(
    "Human-review rows:",
    len(human_review_dataframe),
)

print("\nHuman-review columns:")
print(
    human_review_dataframe
    .columns
    .tolist()
)


# ============================================================
# 10. 检查人工复核列
# ============================================================

human_metric_columns = [
    "human_tool_selection_correct",
    "human_faithful",
    "human_snapshot_correct",
    "human_citation_support",
    "human_uncertainty",
    "human_constraint_satisfaction",
    "human_overall_pass",
]

missing_human_columns = [
    column_name
    for column_name in human_metric_columns
    if column_name
    not in human_review_dataframe.columns
]

if missing_human_columns:
    raise RuntimeError(
        "人工复核文件缺少以下列："
        + ", ".join(
            missing_human_columns
        )
    )


# ============================================================
# 11. 转换人工评分
# ============================================================

for column_name in human_metric_columns:

    human_review_dataframe[
        column_name
    ] = (
        human_review_dataframe[
            column_name
        ]
        .apply(parse_binary_score)
        .astype("Int64")
    )


display_columns = [
    column_name
    for column_name in [
        "question_id",
        "question",
        *human_metric_columns,
        "human_notes",
    ]
    if column_name
    in human_review_dataframe.columns
]

print("\nConverted human-review scores:")

display(
    human_review_dataframe[
        display_columns
    ]
)


# ============================================================
# 12. 生成GPT-6.1人工复核摘要
# ============================================================

human_summary_rows = []

for metric in human_metric_columns:

    valid_values = (
        human_review_dataframe[metric]
        .dropna()
        .astype(int)
    )

    print(
        f"{metric}: "
        f"{len(valid_values)} valid scores"
    )

    if valid_values.empty:
        continue

    human_summary_rows.append({
        "metric": metric,
        "valid_questions": int(
            len(valid_values)
        ),
        "passed_questions": int(
            valid_values.sum()
        ),
        "pass_rate": float(
            valid_values.mean()
        ),
        "tested_model": TESTED_MODEL,
    })


human_summary = pd.DataFrame(
    human_summary_rows
)

if human_summary.empty:
    raise RuntimeError(
        "人工复核文件存在，但没有读取到有效评分。\n"
        "请检查human_*列是否填写了0/1、"
        "TRUE/FALSE或Yes/No。"
    )


human_review_csv_path = (
    RUN_DIR
    / f"human_review_10_{MODEL_FILE_LABEL}.csv"
)

human_review_xlsx_path = (
    RUN_DIR
    / f"human_review_10_{MODEL_FILE_LABEL}.xlsx"
)

human_summary_csv_path = (
    RUN_DIR
    / f"human_summary_{MODEL_FILE_LABEL}.csv"
)

human_summary_xlsx_path = (
    RUN_DIR
    / f"human_summary_{MODEL_FILE_LABEL}.xlsx"
)

human_review_dataframe.to_csv(
    human_review_csv_path,
    index=False,
    encoding="utf-8-sig",
)

human_review_dataframe.to_excel(
    human_review_xlsx_path,
    index=False,
)

human_summary.to_csv(
    human_summary_csv_path,
    index=False,
    encoding="utf-8-sig",
)

human_summary.to_excel(
    human_summary_xlsx_path,
    index=False,
)

print("\nHuman-review summary:")
display(human_summary)


# ============================================================
# 13. 生成Judge和人工评估分歧指标
# ============================================================

comparison_pairs = {
    "faithfulness_disagreement": (
        "judge_faithful",
        "human_faithful",
    ),
    "snapshot_disagreement": (
        "judge_snapshot_correct",
        "human_snapshot_correct",
    ),
    "citation_disagreement": (
        "judge_citation_support",
        "human_citation_support",
    ),
    "uncertainty_disagreement": (
        "judge_uncertainty",
        "human_uncertainty",
    ),
    "constraint_disagreement": (
        "judge_constraint_satisfaction",
        "human_constraint_satisfaction",
    ),
}

comparison_summary_rows = []

for (
    disagreement_name,
    (
        judge_column,
        human_column,
    ),
) in comparison_pairs.items():

    if (
        judge_column
        not in human_review_dataframe.columns
        or human_column
        not in human_review_dataframe.columns
    ):
        continue

    judge_values = (
        human_review_dataframe[
            judge_column
        ]
        .apply(parse_binary_score)
    )

    human_values = (
        human_review_dataframe[
            human_column
        ]
        .apply(parse_binary_score)
    )

    valid_mask = (
        judge_values.notna()
        & human_values.notna()
    )

    if not valid_mask.any():
        continue

    disagreement_values = (
        judge_values[valid_mask]
        .astype(int)
        .ne(
            human_values[valid_mask]
            .astype(int)
        )
    )

    human_review_dataframe[
        disagreement_name
    ] = pd.NA

    human_review_dataframe.loc[
        valid_mask,
        disagreement_name,
    ] = (
        disagreement_values
        .astype(int)
        .values
    )

    comparison_summary_rows.append({
        "metric": disagreement_name,
        "valid_questions": int(
            valid_mask.sum()
        ),
        "disagreement_count": int(
            disagreement_values.sum()
        ),
        "disagreement_rate": float(
            disagreement_values.mean()
        ),
        "tested_model": TESTED_MODEL,
    })


comparison_summary = pd.DataFrame(
    comparison_summary_rows
)

if not comparison_summary.empty:

    comparison_summary.to_csv(
        RUN_DIR
        / f"judge_human_comparison_{MODEL_FILE_LABEL}.csv",
        index=False,
        encoding="utf-8-sig",
    )

    comparison_summary.to_excel(
        RUN_DIR
        / f"judge_human_comparison_{MODEL_FILE_LABEL}.xlsx",
        index=False,
    )

    # 重新保存包含分歧列的人工复核明细
    human_review_dataframe.to_csv(
        human_review_csv_path,
        index=False,
        encoding="utf-8-sig",
    )

    human_review_dataframe.to_excel(
        human_review_xlsx_path,
        index=False,
    )

    print("\nJudge-human disagreement summary:")
    display(comparison_summary)


# ============================================================
# 14. 保存Agent指令
# ============================================================

agent_instructions = globals().get(
    "AGENT_INSTRUCTIONS",
    "[DATA NOT AVAILABLE]",
)

agent_instructions_path = (
    RUN_DIR
    / f"agent_instructions_{MODEL_FILE_LABEL}.txt"
)

agent_instructions_path.write_text(
    str(agent_instructions),
    encoding="utf-8",
)


# ============================================================
# 15. 保存工具审计日志
# ============================================================

tool_audit_log = globals().get(
    "TOOL_AUDIT_LOG",
    [],
)

audit_log_path = (
    RUN_DIR
    / f"tool_audit_log_{MODEL_FILE_LABEL}.json"
)

audit_log_path.write_text(
    json.dumps(
        tool_audit_log,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)


# ============================================================
# 16. 保存实验配置
# ============================================================

metadata = {
    "experiment_type": (
        "baseline_model_evaluation"
    ),
    "tested_model": TESTED_MODEL,
    "model_file_label": MODEL_FILE_LABEL,
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,

    "question_count": int(
        len(evaluation_questions)
    ),
    "evaluation_count": int(
        len(evaluation_corrected)
    ),
    "human_review_count": int(
        len(human_review_dataframe)
    ),

    "corpus_name": globals().get(
        "CORPUS_NAME",
        "[DATA NOT AVAILABLE]",
    ),
    "snapshot_date": globals().get(
        "SNAPSHOT_DATE",
        "[DATA NOT AVAILABLE]",
    ),
    "embedding_model": globals().get(
        "EMBEDDING_MODEL",
        "[DATA NOT AVAILABLE]",
    ),
    "top_k": globals().get(
        "TOP_K",
        "[DATA NOT AVAILABLE]",
    ),
    "min_similarity": globals().get(
        "MIN_SIMILARITY",
        "[DATA NOT AVAILABLE]",
    ),
    "temperature": globals().get(
        "TEMPERATURE",
        0,
    ),

    "tools": [
        tool_name
        for tool_name in [
            "search_knowledge_base",
            "get_route_information",
            "fetch_live_official_page",
            "search_live_official_web",
        ]
        if tool_name in globals()
    ],

    "tool_matching_rule": (
        "Expected tools must be a subset "
        "of actual tools. Tool order is ignored "
        "and additional tools are permitted."
    ),

    "tool_delimiters_supported": [
        ",",
        "|",
        ";",
    ],

    "human_review_source": str(
        HUMAN_REVIEW_PATH
    ),

    "python_version": (
        platform.python_version()
    ),

    "missing_measurements": {
        "average_response_latency": (
            "[DATA NOT AVAILABLE]"
        ),
        "average_api_cost_per_question": (
            "[DATA NOT AVAILABLE]"
        ),
        "end_user_usability_score": (
            "[DATA NOT AVAILABLE]"
        ),
    },
}

metadata_path = (
    RUN_DIR / "baseline_metadata.json"
)

metadata_path.write_text(
    json.dumps(
        metadata,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)


# ============================================================
# 17. 生成统一总摘要
# ============================================================

combined_summary_parts = []

if not automatic_summary.empty:

    automatic_combined = (
        automatic_summary.copy()
    )

    automatic_combined.insert(
        0,
        "evaluation_source",
        "automatic",
    )

    combined_summary_parts.append(
        automatic_combined
    )


if not human_summary.empty:

    human_combined = (
        human_summary.copy()
    )

    human_combined.insert(
        0,
        "evaluation_source",
        "human",
    )

    combined_summary_parts.append(
        human_combined
    )


if combined_summary_parts:

    combined_summary = pd.concat(
        combined_summary_parts,
        ignore_index=True,
        sort=False,
    )

    combined_summary.to_csv(
        RUN_DIR
        / f"combined_summary_{MODEL_FILE_LABEL}.csv",
        index=False,
        encoding="utf-8-sig",
    )

    combined_summary.to_excel(
        RUN_DIR
        / f"combined_summary_{MODEL_FILE_LABEL}.xlsx",
        index=False,
    )

    print("\nCombined summary:")
    display(combined_summary)


# ============================================================
# 18. 创建文件清单
# ============================================================

manifest_rows = []

for file_path in sorted(
    RUN_DIR.iterdir()
):

    if not file_path.is_file():
        continue

    manifest_rows.append({
        "file_name": file_path.name,
        "file_size_bytes": (
            file_path.stat().st_size
        ),
    })


file_manifest = pd.DataFrame(
    manifest_rows
)

file_manifest.to_csv(
    RUN_DIR / "file_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 19. 将实验文件夹压缩成ZIP
# ============================================================

zip_base_path = (
    EXPERIMENTS_DIR
    / EXPERIMENT_NAME
)

zip_path = shutil.make_archive(
    str(zip_base_path),
    "zip",
    root_dir=RUN_DIR,
)


# ============================================================
# 20. 最终检查
# ============================================================

saved_files = pd.DataFrame({
    "file_name": sorted(
        file_path.name
        for file_path in RUN_DIR.iterdir()
        if file_path.is_file()
    )
})

print("\n" + "=" * 72)
print("GPT-6.1 experiment saved successfully")
print("=" * 72)

print("Tested model:", TESTED_MODEL)
print("Experiment folder:", RUN_DIR)
print("ZIP archive:", zip_path)

print("\nSaved files:")
display(saved_files)